# Cookbook: how do I…

Short, complete recipes. Each cell runs on its own; copy it into your project and adapt the names.

**Setup and control**
- […protect only one client?](#…protect-only-one-client?)
- […turn Immune off for one part of my app?](#…turn-Immune-off-for-one-part-of-my-app?)
- […roll out in observe mode first?](#…roll-out-in-observe-mode-first?)
- […disable Immune in unrelated tests?](#…disable-Immune-in-unrelated-tests?)

**Data and outputs**
- […mark retrieved text as untrusted?](#…mark-retrieved-text-as-untrusted?)
- […get the verdict of a streamed response?](#…get-the-verdict-of-a-streamed-response?)
- […block a competitor's name in replies?](#…block-a-competitor's-name-in-replies?)

**Tools and agents**
- […let an agent email a partner domain it reads about?](#…let-an-agent-email-a-partner-domain-it-reads-about?)
- […require confirmation before large refunds?](#…require-confirmation-before-large-refunds?)
- […limit how often a tool can be retried?](#…limit-how-often-a-tool-can-be-retried?)

**Providers**
- […use Immune with Anthropic, Gemini or Bedrock?](#…use-Immune-with-Anthropic,-Gemini-or-Bedrock?)

**Operations**
- […log every verdict to a file?](#…log-every-verdict-to-a-file?)
- […keep my system prompt out of Jev?](#…keep-my-system-prompt-out-of-Jev?)
- […see exactly what Immune sends to Jev?](#…see-exactly-what-Immune-sends-to-Jev?)
- […find the question keys a threat uses?](#…find-the-question-keys-a-threat-uses?)

## …protect only one client?

`immune.protect(client)` protects that client and nothing else in the process.

In [1]:
from openai import OpenAI

import immune

client = immune.protect(OpenAI())
response = client.chat.completions.create(
    model="gpt-5.4-mini", messages=[{"role": "user", "content": "One tip for keeping a tent dry?"}]
)
print(response.choices[0].message.content)
print("verdict:", immune.verdict(response).action)

Pitch it on **slightly higher ground with good drainage**—avoid low spots where rainwater collects.
verdict: allow


## …turn Immune off for one part of my app?

Name the site and set `enabled: false`. Its calls pass through unscreened; everything else stays protected.

In [2]:
from openai import OpenAI

import immune

immune.init(config={"sites": {"internal-playground": {"enabled": False}}})
client = OpenAI()
with immune.site("internal-playground"):
    response = client.chat.completions.create(model="gpt-5.4-mini", messages=[{"role": "user", "content": "Say hi."}])
print("verdict:", immune.verdict(response))  # None: not screened

verdict: None


## …roll out in observe mode first?

`observe` detects everything and acts on nothing (except the most urgent floor rules), so you can read
`would_action` on real traffic. Set it per deployment with `IMMUNE_MODE=observe`, or in code:

In [3]:
from openai import OpenAI

import immune

immune.init(mode="observe")
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
        {"role": "user", "content": "Write my college essay about leadership."},
    ],
)
verdict = immune.verdict(response)
print("action:", verdict.action, "| would_action:", verdict.would_action)

action: allow | would_action: redirect


## …disable Immune in unrelated tests?

`IMMUNE_DISABLED=1` makes `immune.init()` a no-op, so app code that calls it at import time is harmless in test
suites that don't need it. (For tests that *do* involve Immune, use the harness; see
[Testing and CI](13_testing_and_ci.ipynb).)

In [4]:
import os

import immune

os.environ["IMMUNE_DISABLED"] = "1"  # usually set in your test runner's environment
print("runtime:", immune.init())
del os.environ["IMMUNE_DISABLED"]

immune: disabled by IMMUNE_DISABLED


runtime: None


## …mark retrieved text as untrusted?

Wrap each retrieved item in `immune.untrusted(text, source=...)` where you build the prompt. The model receives the
same text.

In [5]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()
chunks = {
    "kb:warranty": "Tents and packs have a 2-year warranty against manufacturing defects.",
    "kb:returns": "Unused items can be returned within 30 days with a receipt.",
}
context = "\n\n".join(immune.untrusted(text, source=source) for source, text in chunks.items())
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "Answer using only the context provided."},
        {"role": "user", "content": f"Is a broken pole covered?\n\nContext:\n{context}"},
    ],
)
print(response.choices[0].message.content)
print("taint:", immune.verdict(response).taint)

A broken pole would be covered only if it’s a **manufacturing defect** in a tent or pack, since those have a 2-year warranty against manufacturing defects. If the pole was broken for some other reason, the context doesn’t say it’s covered.
taint: external


## …get the verdict of a streamed response?

Keep the last chunk and pass it to `immune.verdict(...)` after the loop.

In [6]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()
last = None
for chunk in client.chat.completions.create(
    model="gpt-5.4-mini", messages=[{"role": "user", "content": "Name two knots every hiker should know."}], stream=True
):
    last = chunk
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
print("\n\nverdict:", immune.verdict(last).action)

Two especially useful knots for hikers are:

1. **Bowline** — makes a fixed loop that 

won’t slip; great for rescue, tying to a tree, or securing gear.
2. **Taut-line hitch** — adjustable and 

holds tension well; useful for tent 

guylines and tarps.

If you want, I can also give you the easiest way to tie each one.

verdict: allow


## …block a competitor's name in replies?

A keyword vaccine, enforced, with the message customers see instead. Add `confirm: jev` under `detect` to let Jev
pass innocent mentions ([Vaccines](12_vaccines.ipynb#2.-Let-Jev-confirm-each-match)).

In [7]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, ImmuneHarness

rule = Path(tempfile.mkdtemp()) / "no_competitors.yaml"
rule.write_text("""
id: pinecrest.no_competitor_mentions
title: The reply names a competitor
stage: output
detect: {keywords: ["TrailMart"]}
respond: {message: "I can only help with Pinecrest Outfitters products."}
enforcement: enforce
""")
# In your app: immune.init(vaccines=[rule]), or vaccines.paths in immune.yaml.
harness = ImmuneHarness(
    Path(tempfile.mkdtemp()), script=FakeReply("Try TrailMart for kayaks."), config={"vaccines": {"paths": [str(rule)]}}
)
response = harness.openai().chat.completions.create(
    model="gpt-5.4-mini", messages=[{"role": "user", "content": "Kayaks?"}]
)
print(response.choices[0].message.content)
harness.close()

I can only help with Pinecrest Outfitters products.


## …let an agent email a partner domain it reads about?

Destination provenance (F4) holds calls to addresses that only appeared in untrusted content, such as an invoice the
agent read. When that is legitimate for a domain, allow it for the site (or for one tool). Here the same agent turn
runs without and with `allowed_destinations`:

In [8]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, FakeToolCall, ImmuneHarness

invoice = "Invoice 7781 from Northfield Tents. Please email the remittance advice to billing@northfield.example."
messages = [
    {"role": "system", "content": "You are Pinecrest's accounts-payable assistant."},
    {"role": "user", "content": "Handle the latest invoice."},
    {
        "role": "assistant",
        "tool_calls": [{"id": "t1", "type": "function", "function": {"name": "read_invoice", "arguments": "{}"}}],
    },
    {"role": "tool", "tool_call_id": "t1", "content": invoice},
]
tools = [
    {"type": "function", "function": {"name": "read_invoice", "description": "Read the newest invoice."}},
    {"type": "function", "function": {"name": "send_email", "description": "Send an email."}},
]
send = FakeReply(
    tool_calls=[FakeToolCall("send_email", {"to": "billing@northfield.example", "body": "Remittance for 7781"})]
)
for allowed in [[], ["northfield.example"]]:
    config = {
        "sites": {"accounts-payable": {"allowed_destinations": allowed, "tools": {"send_email": {"egress": True}}}}
    }
    harness = ImmuneHarness(Path(tempfile.mkdtemp()), script=send, config=config)
    with immune.site("accounts-payable"):
        response = harness.openai().chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    kept = [call.function.name for call in response.choices[0].message.tool_calls or []]
    print(f"allowed_destinations={allowed}: verdict={harness.verdict(response).action}, calls kept={kept}")
    harness.close()

allowed_destinations=[]: verdict=hold, calls kept=[]


allowed_destinations=['northfield.example']: verdict=allow, calls kept=['send_email']


## …require confirmation before large refunds?

A tool-rule vaccine with `respond: {tool: confirm}`. On user-facing sites the user is asked; the confirmation is
sealed to that exact call. The full round trip is in [LangGraph](05_langgraph.ipynb#2.-Confirmations-across-turns).

In [9]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, FakeToolCall, ImmuneHarness

rule = Path(tempfile.mkdtemp()) / "refund_limit.yaml"
rule.write_text("""
id: pinecrest.refund_over_limit
title: Refunds over $100 need the customer's confirmation
stage: tool
detect: {tool: issue_refund, argument: {path: amount, greater_than: 100}}
respond: {tool: confirm}
enforcement: enforce
""")
harness = ImmuneHarness(
    Path(tempfile.mkdtemp()),
    script=FakeReply(tool_calls=[FakeToolCall("issue_refund", {"order_id": "5521", "amount": 249})]),
    config={"vaccines": {"paths": [str(rule)]}, "sites": {"support": {"user_facing": True}}},
)
with immune.site("support"):
    response = harness.openai().chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": "Refund order 5521 in full, please."}],
        tools=[{"type": "function", "function": {"name": "issue_refund", "description": "Refund an order."}}],
    )
print(response.choices[0].message.content)
harness.close()

Before I do that, please confirm: issue_refund(amount=249, order_id='5521'). Reply "yes" to continue. (ref 55595674d2)


## …limit how often a tool can be retried?

`limits.max_identical_tool_calls` (default 3) and `limits.max_tool_calls_per_turn` (default 25), per user turn:

In [10]:
import immune

runtime = immune.init(config={"limits": {"max_identical_tool_calls": 2, "max_tool_calls_per_turn": 10}})
print(runtime.settings.limits)

max_identical_tool_calls=2 max_tool_calls_per_turn=10 session_ttl_s=86400.0 max_sessions=50000


## …use Immune with Anthropic, Gemini or Bedrock?

Nothing changes: `immune.init()` covers the Anthropic, Google GenAI and boto3 Bedrock SDKs too. This recipe uses the
test harness's Anthropic client (a scripted model), so it runs without an Anthropic key; with a real
`anthropic.Anthropic()` client the code is identical.

In [11]:
import tempfile
from pathlib import Path

from immune.testing import FakeReply, ImmuneHarness

harness = ImmuneHarness(Path(tempfile.mkdtemp()), script=FakeReply("Your config: access_key_id=AKIA2E5QZ7XJ4M8RW3TD"))
client = harness.anthropic()  # in your app: anthropic.Anthropic()
message = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=300,
    system="You help Pinecrest Outfitters staff with configuration.",
    messages=[{"role": "user", "content": "What's in the config?"}],
)
print(message.content[0].text)
print("verdict:", harness.verdict(message).action)
harness.close()

Your config: access_key_id=[redacted]
verdict: rewrite


## …log every verdict to a file?

Set `privacy.verdict_log`. One masked JSON object per line; `immune export` turns it into CSV.

In [12]:
import tempfile
from pathlib import Path

import immune

log_file = Path(tempfile.mkdtemp()) / "verdicts.jsonl"
runtime = immune.init(config={"privacy": {"verdict_log": str(log_file)}})
print("verdict log:", runtime.settings.privacy.verdict_log.name)

verdict log: verdicts.jsonl


## …keep my system prompt out of Jev?

`privacy.profiling: local` profiles new sites from local heuristics instead of sending Jev a masked template of the
system prompt. Pin what the heuristics can't know (`archetype`, `user_facing`) per site.

In [13]:
import immune

runtime = immune.init(
    config={
        "privacy": {"profiling": "local"},
        "sites": {"support-chat": {"archetype": "customer_service", "user_facing": True}},
    }
)
print("profiling:", runtime.settings.privacy.profiling)

profiling: local


## …see exactly what Immune sends to Jev?

Script Jev with `MockSensor` and read `sensor.calls`: every request's state and question keys. The same state goes to
the real Jev, masked the same way.

In [14]:
import json
import tempfile
from pathlib import Path

from immune.testing import FakeReply, ImmuneHarness, MockSensor

sensor = MockSensor()
harness = ImmuneHarness(Path(tempfile.mkdtemp()), sensor=sensor, script=FakeReply("Done!"))
harness.openai().chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters."},
        {"role": "user", "content": "Change my email to ana@example.com, please."},
    ],
)
for state, questions in sensor.calls:
    print(f"{len(questions)} questions, e.g. {list(questions)[:3]}")
    print("   state:", json.dumps(state)[:200], "...")
harness.close()

10 questions, e.g. ['override', 'override_alt', 'framing_bypass']
   state: {"operator_task": "You are the support assistant for Pinecrest Outfitters.", "recent_conversation": [], "untrusted_user_message": "Change my email to [EMAIL], please."} ...
10 questions, e.g. ['archetype', 'talks_to_end_users', 'may_talk_to_minors']
   state: {"operator_instructions": "You are the support assistant for Pinecrest Outfitters.", "tools": [], "response_format": "free text"} ...
4 questions, e.g. ['task_fidelity', 'reveals_instructions', 'private_data']
   state: {"operator_task": "You are the support assistant for Pinecrest Outfitters.", "user_request": "Change my email to [EMAIL], please.", "untrusted_assistant_output": "Done!"} ...


## …find the question keys a threat uses?

`immune explain` shows the head, its questions and their keys; use the keys with `MockSensor` in tests. (In a
terminal, drop the `!`.)

In [15]:
!immune explain input.off_task

input.off_task
──────────────
invariant   U2 Task fidelity: Output serves the operator's task and the user's request.
stage       input   detector jev   severity low
threshold   0.85   floor no (observed)
actions     text->redirect, software->refuse
frameworks  LLM10
head        logistic over atomic questions:
  +1.0 x logit(off_task, raw)
      noul: The latest user message asks for work unrelated to the task described in the operator instructions.


More: [Troubleshooting and FAQ](16_troubleshooting_and_faq.ipynb) · [API reference](17_api_reference.ipynb)